# GNN Job Recommendation — Build Graph + Train
Pipeline: Fix data → Build HeteroData → Train HeteroSAGE → Evaluate

In [1]:
pip install pandas numpy matplotlib seaborn scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install torch

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install torch_geometric

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from torch_geometric.data import HeteroData
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.nn import SAGEConv, HeteroConv

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

f:\OneDrive\TAI_LIEU_HK8\PBL\job-recommendation-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


## Bước 1 — Load & Fix data còn sót

In [5]:
df_job  = pd.read_csv('../../data/processed/COMBINED_DATA_PROCESSED.csv', encoding='utf-8-sig', low_memory=False)
df_user = pd.read_csv('../../data/processed/USER_DATA_PROCESSED.csv',    encoding='utf-8-sig', low_memory=False)

print(f'Job:  {df_job.shape}  | User: {df_user.shape}')

# --- Fix Age outlier (vẫn còn Age=0, Age=121) ---
df_user['Age'] = df_user['Age'].clip(16, 65)

# --- Fix salary_max User: 92% NaN → fill bằng salary_min * 1.2 ---
df_user['salary_max'] = df_user['salary_max'].fillna(df_user['salary_min'] * 1.2)
df_user['salary_max'] = df_user['salary_max'].fillna(df_user['salary_min'])  # fallback nếu min cũng NaN

# --- Đảm bảo job_function không NaN (dùng để encode) ---
df_job['job_function'] = df_job['job_function'].fillna('Nhân viên')
df_job['company_name'] = df_job['company_name'].fillna('Không rõ')

print('Fix xong. User salary_max nulls:', df_user['salary_max'].isnull().sum())
print('User Age range:', df_user['Age'].min(), '-', df_user['Age'].max())

Job:  (39036, 15)  | User: (3983, 22)
Fix xong. User salary_max nulls: 0
User Age range: 16 - 65


## Bước 2 — TF-IDF embedding (fit chung Job + User)

In [6]:
# Đảm bảo cột text tồn tại
if 'job_text' not in df_job.columns:
    raise ValueError('Thiếu cột job_text trong COMBINED_DATA_PROCESSED. Chạy lại pipeline job.')

df_user['user_text_clean'] = df_user['user_text_clean'].fillna(
    df_user['Skills'].fillna('') + ' ' + df_user['Target'].fillna('')
)

# Fit TF-IDF trên CẢ HAI corpus
all_texts = pd.concat([
    df_job['job_text'].fillna(''),
    df_user['user_text_clean'].fillna('')
], ignore_index=True)

tfidf = TfidfVectorizer(
    max_features=256,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=3,
    max_df=0.9,
)
tfidf.fit(all_texts)

job_text_emb  = tfidf.transform(df_job['job_text'].fillna('')).toarray()          # (N_job, 256)
user_text_emb = tfidf.transform(df_user['user_text_clean'].fillna('')).toarray()  # (N_user, 256)

print(f'Job text emb:  {job_text_emb.shape}')
print(f'User text emb: {user_text_emb.shape}')

Job text emb:  (39036, 256)
User text emb: (3983, 256)


## Bước 3 — Xây dựng Node Feature Matrix X_job và X_user

In [7]:
# ── Encoders: FIT trên Job, TRANSFORM trên cả Job lẫn User ──────────
le_ind  = LabelEncoder()
le_prov = LabelEncoder()
le_emp  = LabelEncoder()
le_func = LabelEncoder()

# Fit trên tập hợp của CẢ HAI để tránh unseen label
all_industries = pd.concat([df_job['industry_group'], df_user['industry_group']]).fillna('Khác').unique()
all_provinces  = pd.concat([df_job['province'],       df_user['province']]).fillna('Khác').unique()

le_ind.fit(all_industries)
le_prov.fit(all_provinces)
le_emp.fit(df_job['employment_type'].fillna('full_time'))
le_func.fit(df_job['job_function'].fillna('Nhân viên'))

# ── X_job ────────────────────────────────────────────────────────────
scaler_job = RobustScaler()
X_job_num = scaler_job.fit_transform(
    df_job[['salary_min','salary_max','exp_min','exp_max']].fillna(0)
)  # (N_job, 4)

job_cat = np.column_stack([
    df_job['salary_type'].fillna(3).values,
    le_emp.transform(df_job['employment_type'].fillna('full_time')),
    le_func.transform(df_job['job_function'].fillna('Nhân viên')),
    le_ind.transform(df_job['industry_group'].fillna('Khác')),
    le_prov.transform(df_job['province'].fillna('Khác')),
])  # (N_job, 5)

X_job = np.hstack([X_job_num, job_cat, job_text_emb]).astype(np.float32)  # (N_job, 265)
print(f'X_job shape: {X_job.shape}')

# ── X_user ───────────────────────────────────────────────────────────
le_gender = LabelEncoder()
le_degree = LabelEncoder()
le_marr   = LabelEncoder()

# Degree: extract cấp học từ text thô
def extract_degree(text):
    if pd.isna(text): return 'other'
    t = str(text).lower()
    if any(k in t for k in ['thạc sĩ', 'master']):        return 'master'
    if any(k in t for k in ['tiến sĩ', 'phd']):           return 'phd'
    if any(k in t for k in ['đại học', 'university']):    return 'university'
    if any(k in t for k in ['cao đẳng', 'college']):      return 'college'
    if any(k in t for k in ['trung cấp', 'dạy nghề']):    return 'vocational'
    if any(k in t for k in ['thpt', 'trung học']):        return 'highschool'
    return 'other'

df_user['degree_level'] = df_user['Degree'].apply(extract_degree)

le_gender.fit(df_user['Gender'].fillna('Khác'))
le_degree.fit(df_user['degree_level'])
le_marr.fit(df_user['Marriage'].fillna('Khác'))

scaler_user = RobustScaler()
X_user_num = scaler_user.fit_transform(
    df_user[['Age','exp_min','exp_max','salary_min','salary_max']].fillna(0)
)  # (N_user, 5)

user_cat = np.column_stack([
    le_gender.transform(df_user['Gender'].fillna('Khác')),
    le_degree.transform(df_user['degree_level']),
    le_marr.transform(df_user['Marriage'].fillna('Khác')),
    le_ind.transform(df_user['industry_group'].fillna('Khác')),
    le_prov.transform(df_user['province'].fillna('Khác')),
    df_user['salary_type'].fillna(3).values,
])  # (N_user, 6)

X_user = np.hstack([X_user_num, user_cat, user_text_emb]).astype(np.float32)  # (N_user, 267)
print(f'X_user shape: {X_user.shape}')

X_job shape: (39036, 265)
X_user shape: (3983, 267)


## Bước 4 — Xây dựng Edges và HeteroData

In [8]:
np.random.seed(42)
df_job  = df_job.reset_index(drop=True)
df_user = df_user.reset_index(drop=True)

user_ids, job_ids = [], []

for uid in range(len(df_user)):
    u = df_user.iloc[uid]

    # Ưu tiên 1: cùng ngành + cùng tỉnh
    mask = (
        (df_job['industry_group'] == u['industry_group']) &
        (df_job['province']       == u['province'])
    )
    matched = df_job[mask].index.tolist()

    # Fallback: chỉ cùng ngành
    if len(matched) < 3:
        matched = df_job[df_job['industry_group'] == u['industry_group']].index.tolist()

    # Fallback cuối: random
    if len(matched) < 1:
        matched = df_job.sample(5).index.tolist()

    sampled = np.random.choice(matched, size=min(10, len(matched)), replace=False)
    user_ids.extend([uid] * len(sampled))
    job_ids.extend(sampled.tolist())

edge_index = torch.tensor([user_ids, job_ids], dtype=torch.long)
print(f'Tổng edges (user→job): {edge_index.shape[1]:,}')
print(f'Avg edges/user: {edge_index.shape[1]/len(df_user):.1f}')

# --- Tạo HeteroData ---
data = HeteroData()
data['user'].x    = torch.tensor(X_user, dtype=torch.float)
data['job'].x     = torch.tensor(X_job,  dtype=torch.float)
data['user', 'applies', 'job'].edge_index = edge_index

# Thêm node_id để trace về sau
data['user'].node_id = torch.arange(len(df_user))
data['job'].node_id  = torch.arange(len(df_job))

print(data)

Tổng edges (user→job): 38,370
Avg edges/user: 9.6
HeteroData(
  user={
    x=[3983, 267],
    node_id=[3983],
  },
  job={
    x=[39036, 265],
    node_id=[39036],
  },
  (user, applies, job)={ edge_index=[2, 38370] }
)


## Bước 5 — Train/Val/Test Split

In [9]:
transform = RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    is_undirected=False,
    neg_sampling_ratio=1.0,         # 1 negative per positive
    add_negative_train_samples=True,
    edge_types=[('user', 'applies', 'job')],
    rev_edge_types=[('job', 'rev_applies', 'user')],
)
train_data, val_data, test_data = transform(data)

print('Train edges:', train_data['user','applies','job'].edge_label_index.shape[1])
print('Val edges:  ', val_data['user','applies','job'].edge_label_index.shape[1])
print('Test edges: ', test_data['user','applies','job'].edge_label_index.shape[1])

# Lưu để dùng lại
torch.save(train_data, '../../data/train_data.pt')
torch.save(val_data,   '../../data/val_data.pt')
torch.save(test_data,  '../../data/test_data.pt')


Train edges: 61392
Val edges:   7674
Test edges:  7674


## Bước 6 — Định nghĩa Model

In [10]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, HeteroConv

class GNNEncoder(torch.nn.Module):
    def __init__(self, hidden_channels: int, out_channels: int, dropout: float = 0.3):
        super().__init__()
        self.dropout = dropout

        # Linear projection: đảm bảo mọi node đều có embedding
        # dù không có neighbor nào (isolated nodes)
        self.proj = torch.nn.ModuleDict({
            'user': torch.nn.LazyLinear(hidden_channels),
            'job':  torch.nn.LazyLinear(hidden_channels),
        })

        self.conv1 = HeteroConv({
            ('user', 'applies',     'job'): SAGEConv((-1, -1), hidden_channels),
            ('job',  'rev_applies', 'user'): SAGEConv((-1, -1), hidden_channels),
        }, aggr='mean')

        self.conv2 = HeteroConv({
            ('user', 'applies',     'job'): SAGEConv((-1, -1), out_channels),
            ('job',  'rev_applies', 'user'): SAGEConv((-1, -1), out_channels),
        }, aggr='mean')

        self.out_proj = torch.nn.ModuleDict({
            'user': torch.nn.Linear(hidden_channels, out_channels),
            'job':  torch.nn.Linear(hidden_channels, out_channels),
        })

    def forward(self, x_dict, edge_index_dict):
        # Bước 0: Project input → hidden_channels làm fallback
        # Đây là embedding "chắc chắn có" cho mọi node
        proj_dict = {k: F.relu(self.proj[k](v)) for k, v in x_dict.items()}

        # Bước 1: Conv layer 1
        conv1_out = self.conv1(x_dict, edge_index_dict)

        # Merge: node nào có neighbor dùng conv output,
        # node isolated dùng proj_dict làm fallback
        x_dict_1 = {}
        for k in proj_dict:
            if k in conv1_out and conv1_out[k] is not None:
                x_dict_1[k] = F.relu(conv1_out[k])
            else:
                x_dict_1[k] = proj_dict[k]  # fallback

        x_dict_1 = {k: F.dropout(v, p=self.dropout, training=self.training)
                    for k, v in x_dict_1.items()}

        # Bước 2: Conv layer 2
        conv2_out = self.conv2(x_dict_1, edge_index_dict)

        # Merge lần 2: fallback = out_proj(x_dict_1)
        x_dict_2 = {}
        for k in x_dict_1:
            if k in conv2_out and conv2_out[k] is not None:
                x_dict_2[k] = conv2_out[k]
            else:
                x_dict_2[k] = self.out_proj[k](x_dict_1[k])  # fallback

        return x_dict_2


class EdgePredictor(torch.nn.Module):
    def __init__(self, in_channels: int):
        super().__init__()
        self.lin = torch.nn.Sequential(
            torch.nn.Linear(in_channels * 2, in_channels),
            torch.nn.ReLU(),
            torch.nn.Linear(in_channels, 1),
        )

    def forward(self, z_user, z_job, edge_label_index):
        u = z_user[edge_label_index[0]]
        j = z_job[edge_label_index[1]]
        return self.lin(torch.cat([u, j], dim=-1)).squeeze(-1)


class JobRecModel(torch.nn.Module):
    def __init__(self, hidden_channels: int = 128, out_channels: int = 64, dropout: float = 0.3):
        super().__init__()
        self.encoder = GNNEncoder(hidden_channels, out_channels, dropout)
        self.decoder = EdgePredictor(out_channels)

    def forward(self, x_dict, edge_index_dict, edge_label_index):
        z_dict = self.encoder(x_dict, edge_index_dict)
        return self.decoder(z_dict['user'], z_dict['job'], edge_label_index)



In [11]:
# ── Warm-up ─────────────────────────────────────────────────────────
def add_reverse_edges(data):
    """Thêm rev_applies nếu chưa có."""
    if ('job', 'rev_applies', 'user') not in data.edge_types:
        ei = data['user', 'applies', 'job'].edge_index
        data['job', 'rev_applies', 'user'].edge_index = ei.flip(0)
    return data

train_data = add_reverse_edges(train_data)
val_data   = add_reverse_edges(val_data)
test_data  = add_reverse_edges(test_data)

model = JobRecModel(hidden_channels=128, out_channels=64, dropout=0.3).to(device)

model.eval()
with torch.no_grad():
    _td = train_data.to(device)
    _ = model(
        _td.x_dict,
        _td.edge_index_dict,
        _td['user', 'applies', 'job'].edge_label_index[:, :4]
    )

print("Warm-up thành công!")



Warm-up thành công!


In [12]:
import torch.nn as nn

total_params = 0
for p in model.parameters():
    if isinstance(p, nn.parameter.UninitializedParameter):
        continue 
    if p.requires_grad:
        total_params += p.numel()

print(f"Tổng số tham số đã khởi tạo: {total_params:,}")

Tổng số tham số đã khởi tạo: 178,049


## Bước 7 — Hàm Train & Evaluate

In [14]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5
)

# Đưa data lên device
train_data = train_data.to(device)
val_data   = val_data.to(device)
test_data  = test_data.to(device)

EDGE_TYPE = ('user', 'applies', 'job')


def train():
    """1 epoch training. Trả về loss."""
    model.train()
    optimizer.zero_grad()

    logits = model(
        train_data.x_dict,
        train_data.edge_index_dict,
        train_data[EDGE_TYPE].edge_label_index,
    )
    labels = train_data[EDGE_TYPE].edge_label.float()

    loss = F.binary_cross_entropy_with_logits(logits, labels)
    loss.backward()
    # Gradient clipping: tránh exploding gradient
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    return float(loss)


@torch.no_grad()
def evaluate(data, split_name='Val'):
    """Tính AUC, Precision, Recall, F1 trên tập data."""
    model.eval()

    logits = model(
        data.x_dict,
        data.edge_index_dict,
        data[EDGE_TYPE].edge_label_index,
    )
    probs  = logits.sigmoid().cpu().numpy()
    preds  = (probs > 0.5).astype(int)
    labels = data[EDGE_TYPE].edge_label.cpu().numpy().astype(int)

    auc  = roc_auc_score(labels, probs)
    prec = precision_score(labels, preds, zero_division=0)
    rec  = recall_score(labels, preds, zero_division=0)
    f1   = f1_score(labels, preds, zero_division=0)

    return {'split': split_name, 'AUC': auc, 'Precision': prec, 'Recall': rec, 'F1': f1}


print('Hàm train() và evaluate() đã sẵn sàng.')

Hàm train() và evaluate() đã sẵn sàng.


## Bước 8 — Training Loop

In [16]:
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
import numpy as np

@torch.no_grad()
def evaluate_comprehensive(model, data, edge_type, k_values=[5, 10, 20]):
    """
    Đánh giá toàn diện với nhiều metrics
    
    Returns:
        dict: {
            'auc': float,
            'precision@k': dict,
            'recall@k': dict,
            'ndcg@k': dict,
            'hit_rate@k': dict
        }
    """
    model.eval()
    
    # Get embeddings
    z_dict = model.encoder(data.x_dict, data.edge_index_dict)
    z_user = z_dict['user']
    z_job = z_dict['job']
    
    # Get edge labels
    edge_label_index = data[edge_type].edge_label_index
    edge_label = data[edge_type].edge_label
    
    # Predictions
    pred = model.decoder(z_user, z_job, edge_label_index).sigmoid()
    
    # 1. AUC
    auc = roc_auc_score(edge_label.cpu(), pred.cpu())
    
    # 2. Precision@K, Recall@K, NDCG@K, Hit Rate@K
    metrics = {
        'auc': auc,
        'precision@k': {},
        'recall@k': {},
        'ndcg@k': {},
        'hit_rate@k': {}
    }
    
    # Group by user
    users = edge_label_index[0].cpu().numpy()
    unique_users = np.unique(users)
    
    for k in k_values:
        precisions = []
        recalls = []
        ndcgs = []
        hits = []
        
        for user_id in unique_users:
            # Get all jobs for this user
            user_mask = users == user_id
            user_jobs = edge_label_index[1][user_mask].cpu().numpy()
            user_labels = edge_label[user_mask].cpu().numpy()
            user_preds = pred[user_mask].cpu().numpy()
            
            # Top-K predictions
            top_k_idx = np.argsort(user_preds)[::-1][:k]
            top_k_labels = user_labels[top_k_idx]
            
            # Precision@K
            precision = top_k_labels.sum() / k
            precisions.append(precision)
            
            # Recall@K
            total_relevant = user_labels.sum()
            recall = top_k_labels.sum() / total_relevant if total_relevant > 0 else 0
            recalls.append(recall)
            
            # NDCG@K
            # NDCG@K
            # Dùng enumerate để duyệt an toàn theo chiều dài thực tế của mảng thay vì ép chạy tới k
            dcg = np.sum([label / np.log2(i + 2) for i, label in enumerate(top_k_labels)])

            ideal_labels = np.sort(user_labels)[::-1][:k]
            idcg = np.sum([label / np.log2(i + 2) for i, label in enumerate(ideal_labels)])

            # Lưu ý: Cần xử lý trường hợp idcg = 0 để tránh lỗi chia cho 0 (ZeroDivisionError)
            ndcg = dcg / idcg if idcg > 0 else 0.0
            ndcgs.append(ndcg)
            
            # Hit Rate@K
            hit = 1 if top_k_labels.sum() > 0 else 0
            hits.append(hit)
        
        metrics[f'precision@{k}'] = np.mean(precisions)
        metrics[f'recall@{k}'] = np.mean(recalls)
        metrics[f'ndcg@{k}'] = np.mean(ndcgs)
        metrics[f'hit_rate@{k}'] = np.mean(hits)
    
    return metrics


best_auc = 0
for epoch in range(1, 151):
    loss = train()
    if epoch % 10 == 0:
        val_metrics = evaluate_comprehensive(model, val_data, EDGE_TYPE)
        print(f'Epoch {epoch:03d}: Loss={loss:.4f} | Val AUC={val_metrics["auc"]:.4f}')
        
        if val_metrics['auc'] > best_auc:
            best_auc = val_metrics['auc']
            torch.save(model.state_dict(), '../../data/best_model1.pt')
            print(f'  ✅ Saved best model (AUC={best_auc:.4f})')

# Sau khi train xong, lưu thêm các artifacts cần thiết
import pickle
torch.save(model.state_dict(), '../../data/best_model1.pt')
with open('../../data/tfidf.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
with open('../../data/preprocessors.pkl', 'wb') as f:
    pickle.dump({
    'scaler_job': scaler_job,
    'scaler_user': scaler_user,
    'le_ind': le_ind,
    'le_prov': le_prov,
    'le_emp': le_emp,
    'le_func': le_func,
    'le_gender': le_gender,
    'le_degree': le_degree,
    'le_marr': le_marr,
}, f)

# Final test
test_metrics = evaluate_comprehensive(model, test_data, EDGE_TYPE)

print("\n" + "="*60)
print("FINAL TEST RESULTS")
print("="*60)
for metric, value in test_metrics.items():
    print(f"{metric:20s}: {value:.4f}")

C:\Users\voquy\AppData\Local\Temp\ipykernel_23508\3167527786.py:31: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:839.)
  return float(loss)


Epoch 010: Loss=232.3159 | Val AUC=0.6702
  ✅ Saved best model (AUC=0.6702)
Epoch 020: Loss=0.8402 | Val AUC=0.7173
  ✅ Saved best model (AUC=0.7173)
Epoch 030: Loss=0.3614 | Val AUC=0.7372
  ✅ Saved best model (AUC=0.7372)
Epoch 040: Loss=0.3296 | Val AUC=0.7607
  ✅ Saved best model (AUC=0.7607)
Epoch 050: Loss=0.3221 | Val AUC=0.7722
  ✅ Saved best model (AUC=0.7722)
Epoch 060: Loss=0.2761 | Val AUC=0.7866
  ✅ Saved best model (AUC=0.7866)
Epoch 070: Loss=0.2342 | Val AUC=0.7958
  ✅ Saved best model (AUC=0.7958)
Epoch 080: Loss=0.2116 | Val AUC=0.8050
  ✅ Saved best model (AUC=0.8050)
Epoch 090: Loss=0.1906 | Val AUC=0.8098
  ✅ Saved best model (AUC=0.8098)
Epoch 100: Loss=0.1810 | Val AUC=0.8157
  ✅ Saved best model (AUC=0.8157)
Epoch 110: Loss=0.1776 | Val AUC=0.8179
  ✅ Saved best model (AUC=0.8179)
Epoch 120: Loss=0.1821 | Val AUC=0.8219
  ✅ Saved best model (AUC=0.8219)
Epoch 130: Loss=0.1756 | Val AUC=0.8237
  ✅ Saved best model (AUC=0.8237)
Epoch 140: Loss=0.1660 | Val AUC=0.8

TypeError: unsupported format string passed to dict.__format__

In [17]:
def ablation_study():
    """
    So sánh các variants của model
    """
    results = []
    
    # 1. Full model (baseline của bạn)
    model_full = model(
        use_text=True,
        use_skills=True,
        use_gnn='sage'
    )
    results.append({
        'name': 'Full GraphSAGE',
        'auc': evaluate(model_full)
    })
    
    # 2. Without text features
    model_no_text = model(
        use_text=False,
        use_skills=True,
        use_gnn='sage'
    )
    results.append({
        'name': 'GraphSAGE (no text)',
        'auc': evaluate(model_no_text)
    })
    
    # 3. Without skill edges
    model_no_skills = model(
        use_text=True,
        use_skills=False,
        use_gnn='sage'
    )
    results.append({
        'name': 'GraphSAGE (no skills)',
        'auc': evaluate(model_no_skills)
    })
    
    # 4. GCN instead of SAGE
    model_gcn = model(
        use_text=True,
        use_skills=True,
        use_gnn='gcn'
    )
    results.append({
        'name': 'GCN (full)',
        'auc': evaluate(model_gcn)
    })
    
    # 5. GAT instead of SAGE
    model_gat = model(
        use_text=True,
        use_skills=True,
        use_gnn='gat'
    )
    results.append({
        'name': 'GAT (full)',
        'auc': evaluate(model_gat)
    })
    
    # Print results
    results_df = pd.DataFrame(results)
    print(results_df.sort_values('auc', ascending=False))
    
    return results_df

In [18]:
@torch.no_grad()
def test_cold_start(model, test_data, num_new_users=100):
    """
    Test model với new users (cold-start)
    
    Strategy:
    - Chọn 100 users random từ test set
    - Giả sử họ là new users (không có history)
    - Chỉ dùng user features (không dùng graph structure)
    - Xem model có recommend tốt không
    """
    model.eval()
    
    # Get all user embeddings
    z_dict = model.encoder(test_data.x_dict, test_data.edge_index_dict)
    z_user = z_dict['user']
    z_job = z_dict['job']
    
    # Random sample new users
    all_users = torch.arange(z_user.shape[0])
    new_users = all_users[torch.randperm(len(all_users))[:num_new_users]]
    
    precisions = []
    
    for user_idx in new_users:
        # Get true positive jobs for this user
        edge_index = test_data[('user', 'applies_to', 'job')].edge_label_index
        edge_label = test_data[('user', 'applies_to', 'job')].edge_label
        
        user_mask = edge_index[0] == user_idx
        true_jobs = edge_index[1][user_mask & (edge_label == 1)]
        
        if len(true_jobs) == 0:
            continue
        
        # Recommend top-10 jobs
        u_emb = z_user[user_idx].unsqueeze(0)
        scores = (u_emb @ z_job.T).squeeze()
        
        top_10 = torch.topk(scores, 10).indices
        
        # Precision
        hits = sum([1 for job in top_10 if job in true_jobs])
        precision = hits / 10
        precisions.append(precision)
    
    print(f"Cold-start Precision@10: {np.mean(precisions):.4f}")
    return np.mean(precisions)

## Bước 9 — Đánh giá cuối trên Test set

In [21]:
# Load lại model tốt nhất
model.load_state_dict(torch.load('../../data/best_model1.pt', map_location=device, weights_only=False))

test_metrics = evaluate(test_data, 'Test')
print('=' * 45)
print('FINAL TEST RESULTS')
print('=' * 45)
for k, v in test_metrics.items():
    if k != 'split':
        print(f'  {k:12s}: {v:.4f}')
print('=' * 45)

FINAL TEST RESULTS
  AUC         : 0.8251
  Precision   : 0.8516
  Recall      : 0.7086
  F1          : 0.7735


## Bước 10 — Hàm Gợi ý việc làm cho 1 User

In [27]:
@torch.no_grad()
def recommend_jobs(user_idx: int, top_k: int = 10):
    """
    Gợi ý top_k công việc phù hợp nhất cho user có index = user_idx.
    Trả về DataFrame gồm: job_title, company_name, industry_group, province, score.
    """
    model.eval()

    # Lấy embedding của toàn bộ nodes từ train_data (chứa toàn bộ nodes)
    z_dict = model.encoder(train_data.x_dict, train_data.edge_index_dict)
    z_user = z_dict['user']  # (N_user, out_channels)
    z_job  = z_dict['job']   # (N_job,  out_channels)

    # Vector của user cần gợi ý
    u_emb = z_user[user_idx].unsqueeze(0)  # (1, out_channels)

    # Tính score với TẤT CẢ jobs
    n_jobs = z_job.shape[0]
    edge_label_index = torch.stack([
        torch.full((n_jobs,), user_idx, dtype=torch.long, device=device),
        torch.arange(n_jobs, device=device)
    ])  # (2, N_job)

    scores = model.decoder(z_user, z_job, edge_label_index).sigmoid().cpu().numpy()

    # Lấy top_k
    top_indices = np.argsort(scores)[::-1][:top_k]

    result = df_job.iloc[top_indices][['job_title', 'company_name', 'industry_group', 'province', 'salary_min', 'salary_max']].copy()
    result['score'] = scores[top_indices].round(4)
    result.index = range(1, top_k + 1)

    user_info = df_user.iloc[user_idx]
    print(f"Gợi ý cho: {user_info['User Name']} | Ngành: {user_info['industry_group']} | Tỉnh: {user_info['province']}")
    print(f"Kinh nghiệm: {user_info['Work Experience']}")
    print()
    return result


# Thử gợi ý cho user đầu tiên
recommend_jobs(user_idx=3105, top_k=10)

Gợi ý cho: Võ Hoàng Chương. | Ngành: Vận tải - Logistics | Tỉnh: Hồ Chí Minh
Kinh nghiệm: Trên 10 năm



,job_title,company_name,industry_group,province,salary_min,salary_max,score
1,Nhân viên Xuất Nhập Khẩu 27397 - KCN Thăng Lon...,Công Ty TNHH Reeracoen Việt Nam,Vận tải - Logistics,Hưng Yên,-0.166667,-0.250000,0.9879
2,Tuyển Dụng 8 lái xe và 7 phụ xe bốc xếp làm tạ...,0364.459.519,Vận tải - Logistics,Hà Nam,-0.166667,-0.083333,0.9871
3,Thủ Kho Nam,Công ty cổ phần xuất nhập khẩu nam thái sơn - ...,Vận tải - Logistics,Hải Dương,-1.000000,-0.500000,0.9864
4,Tuyển 5 lái xe và phụ xe Tại Hưng Yênn,0398-092-263 Công Ty thương mại vận tải,Vận tải - Logistics,Hưng Yên,-0.166667,-0.083333,0.9860
5,"Cần tuyển 5 lái xe ,phụ xe làm tại Hà Nam",0398-092-263 Công Ty thương mại vận tải,Vận tải - Logistics,Hà Nam,-0.166667,-0.083333,0.9859
6,Trưởng Bưu Cục Hải Phòng | Thu nhập đến 25 triệu,Công Ty Cổ Phần Hai Bốn Bảy,Vận tải - Logistics,Hải Phòng,0.666667,0.750000,0.9855
7,Nhân Viên Thủ Kho,CÔNG TY TNHH MTV NBC PACIFIC,Vận tải - Logistics,Hưng Yên,-1.000000,-0.500000,0.9848
8,Tuyển 4 lái xe và phụ xe Tại Hưng Yên,0398-092-263 Công Ty thương mại vận tải,Vận tải - Logistics,Hưng Yên,-0.166667,-0.083333,0.9848
9,Nhân viên giao nhận,Không rõ,Vận tải - Logistics,Hồ Chí Minh,-0.333333,-0.083333,0.9847
10,[Hải Phòng] Warehouse Supervisor,Công Ty TNHH LF Logistics (Việt Nam),Vận tải - Logistics,Hải Phòng,0.333333,0.166667,0.9846
